# Data cleaning

This notebook cleans the HCV dataset. It checks the missing values, compares three ways of filling them, removes the suspected blood donors, fills the gaps with the median, reports the outliers and saves the cleaned file.

## Imports

In [1]:
# standard library
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import MonteCarloMethod, chi2_contingency, fisher_exact
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Configuration

File paths, the random seed and the column lists are set once here.

In [2]:
# input and output files, relative to the project folder
DATA_PATH = Path("hcvdat0.csv")
OUTPUT_PATH = Path("hcv_median_imputed.csv")

# one seed is used everywhere so the results can be reproduced exactly
SEED = 42
ALPHA = 0.05          # significance level for the missingness tests
N_RESAMPLES = 10000   # Monte Carlo resamples per test
N_FOLDS = 5           # cross-validation folds

# order used for every table. "Suspected Blood Donor" is dropped later
CLASS_ORDER = ["Healthy", "Hepatitis", "Fibrosis", "Cirrhosis"]

# the 11 measured variables (Sex is categorical, so it is kept separate)
NUMERIC_COLUMNS = [
    "Age", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL",
    "CREA", "GGT", "PROT",
]

# numeric code for each cleaned label (kept from the original dataset)
CATEGORY_CODES = {
    "Healthy": 0,
    "Suspected Blood Donor": 1,
    "Hepatitis": 2,
    "Fibrosis": 3,
    "Cirrhosis": 4,
}

## Load the raw data

In [3]:
# read the raw file exactly as supplied and look at the first rows
raw_df = pd.read_csv(DATA_PATH)
raw_df.head()

,Unnamed: 0,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT
0,1,0=Blood Donor,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0
1,2,0=Blood Donor,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5
2,3,0=Blood Donor,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3
3,4,0=Blood Donor,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7
4,5,0=Blood Donor,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7


## Tidy the labels

The first column is a row number, so it is dropped. Labels such as `0=Blood Donor` are cut down to the name, and "Blood Donor" is renamed "Healthy". `cat_num` holds the number of each label.

In [4]:
# the first column is only a row number, so it is dropped
df = raw_df.drop(columns=["Unnamed: 0"])

# labels look like "0=Blood Donor"; keep only the text after the "="
df["Category"] = df["Category"].str.replace(r"^[^=]*=", "", regex=True)

# use the clearer names "Healthy" and "Suspected Blood Donor"
df["Category"] = df["Category"].replace({
    "Blood Donor": "Healthy",
    "suspect Blood Donor": "Suspected Blood Donor",
})

# add the numeric label code (a label, not a measurement)
df["cat_num"] = df["Category"].map(CATEGORY_CODES).astype("Int64")

print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
df.head()

Rows: 615, columns: 14


,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,cat_num
0,Healthy,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,0
1,Healthy,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,0
2,Healthy,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,0
3,Healthy,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,0
4,Healthy,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,0


## Class sizes

Most patients are healthy, so the classes are very uneven.

In [5]:
# count the patients in each category
df.groupby(["cat_num", "Category"]).size().rename("Count").reset_index()

,cat_num,Category,Count
0,0,Healthy,533
1,1,Suspected Blood Donor,7
2,2,Hepatitis,24
3,3,Fibrosis,21
4,4,Cirrhosis,30


## Missing values

Number and percentage of missing values in each column.

In [6]:
# count missing values per column and express them as a percentage
missing_summary = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Missing (%)": df.isna().mean().mul(100).round(2),
})
missing_summary

,Missing,Missing (%)
Category,0,0.00
Age,0,0.00
Sex,0,0.00
ALB,1,0.16
ALP,18,2.93
ALT,1,0.16
AST,0,0.00
BIL,0,0.00
CHE,0,0.00
CHOL,10,1.63


### Missing values by category

Percentage of missing values in each column for each diagnosis.

In [7]:
# keep only the columns that actually have missing values
missing_columns = missing_summary.index[missing_summary["Missing"] > 0]

# mark each value as missing (True) or present (False), then average
# within each category to get the percentage missing
missing_by_category = (
    df[missing_columns].isna().groupby(df["Category"]).mean().mul(100)
)
missing_by_category.round(2)

,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Cirrhosis,3.33,20.00,0.00,6.67,3.33
Fibrosis,0.00,42.86,0.00,4.76,0.00
Healthy,0.00,0.00,0.00,1.31,0.00
Hepatitis,0.00,12.50,4.17,0.00,0.00
Suspected Blood Donor,0.00,0.00,0.00,0.00,0.00


## Is missingness related to diagnosis?

A chi-square test checks whether each column is missing more often in some diagnoses than in others. Some groups are very small, so the p-value comes from 10,000 random resamples. The p-values are Bonferroni adjusted because five columns are tested.

In [8]:
# test each incomplete column separately
p_values = []
for column in missing_columns:
    # count missing and present values within each category
    counts = pd.crosstab(df["Category"], df[column].isna())

    # Monte Carlo sampling is used because some counts are very small
    # the seed is restarted for every column so each test is repeatable
    method = MonteCarloMethod(
        n_resamples=N_RESAMPLES, rng=np.random.default_rng(SEED)
    )
    result = chi2_contingency(counts, correction=False, method=method)
    p_values.append(result.pvalue)

missingness = pd.DataFrame({
    "Column": missing_columns,
    "Missing": df[missing_columns].isna().sum().to_numpy(),
    "p_value": p_values,
})

# Bonferroni correction: multiply by the number of tests (maximum of 1)
missingness["adjusted_p"] = (
    missingness["p_value"] * len(missingness)
).clip(upper=1)
missingness["Significant"] = missingness["adjusted_p"] < ALPHA
missingness

,Column,Missing,p_value,adjusted_p,Significant
0,ALB,1,0.130987,0.654935,False
1,ALP,18,0.000100,0.000500,True
2,ALT,1,0.086191,0.430957,False
3,CHOL,10,0.222378,1.000000,False
4,PROT,1,0.130987,0.654935,False


### Findings: missing values

Five columns have missing values: ALB (1), ALP (18), ALT (1), CHOL (10) and PROT (1). Only ALP is linked to diagnosis (adjusted p = 0.0005). ALP is missing for 9 of 21 fibrosis patients, 6 of 30 cirrhosis patients and 3 of 24 hepatitis patients, and for no healthy patients.

Deleting rows with missing values would remove many of the sick patients, so the gaps are filled instead.

## Comparing imputation methods

Three ways of filling the gaps are compared on the same cross-validation folds: the median, the median with a flag for missing values, and iterative imputation. Each is tested with logistic regression and random forest. The seven suspected blood donors are removed first, which leaves 608 patients.

### Four-class dataset

In [9]:
# remove the seven suspected donors; the rest of this section uses
# the remaining 608 patients. The full dataframe df is left unchanged
comparison_df = df[df["Category"] != "Suspected Blood Donor"]
comparison_df = comparison_df.reset_index(drop=True)

# predictors exclude Category and cat_num because they encode the outcome
features = comparison_df[NUMERIC_COLUMNS + ["Sex"]]
target = comparison_df["Category"]

# class sizes after removing the suspected donors
target.value_counts().reindex(CLASS_ORDER).rename("Patients")

Category
Healthy      533
Hepatitis     24
Fibrosis      21
Cirrhosis     30
Name: Patients, dtype: int64

### Missing values in the retained patients

In [10]:
# mark missing numeric values and count them within each category
missing_flags = features[NUMERIC_COLUMNS].isna()
missing_counts = missing_flags.groupby(target).sum().reindex(CLASS_ORDER)

# show only the columns that have at least one missing value
missing_counts.loc[:, missing_flags.any()]

,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Healthy,0,0,0,7,0
Hepatitis,0,3,1,0,0
Fibrosis,0,9,0,1,0
Cirrhosis,1,6,0,2,1


### Missing values by sex

TRIPOD+AI asks whether data quality was similar across groups such as sex. Each incomplete column is compared between men and women with Fisher's exact test, because some counts are very small. The p-values are Bonferroni adjusted for the five columns.

In [11]:
# count the missing values in each incomplete column for men and women
sex = comparison_df["Sex"]
incomplete = missing_flags.columns[missing_flags.any()]
missing_by_sex = missing_flags[incomplete].groupby(sex).sum().T
missing_by_sex.columns = ["Women missing", "Men missing"]

# number of women and men, used for the percentages
n_women = (sex == "f").sum()
n_men = (sex == "m").sum()
missing_by_sex["Women (%)"] = (100 * missing_by_sex["Women missing"] / n_women).round(2)
missing_by_sex["Men (%)"] = (100 * missing_by_sex["Men missing"] / n_men).round(2)

# Fisher's exact test on the 2 x 2 table: sex by missing or present
p_values = []
for column in incomplete:
    women_missing = missing_by_sex.loc[column, "Women missing"]
    men_missing = missing_by_sex.loc[column, "Men missing"]
    table = [[women_missing, n_women - women_missing],
             [men_missing, n_men - men_missing]]
    p_values.append(fisher_exact(table).pvalue)
missing_by_sex["p_value"] = p_values

# Bonferroni correction: multiply by the number of columns (maximum of 1)
missing_by_sex["adjusted_p"] = (
    missing_by_sex["p_value"] * len(incomplete)
).clip(upper=1)
missing_by_sex["Significant"] = missing_by_sex["adjusted_p"] < ALPHA

print(f"Women: {n_women}, men: {n_men}")
missing_by_sex.round(4)

Women: 237, men: 371


,Women missing,Men missing,Women (%),Men (%),p_value,adjusted_p,Significant
ALB,0,1,0.00,0.27,1.0000,1.000,False
ALP,6,12,2.53,3.23,0.8071,1.000,False
ALT,0,1,0.00,0.27,1.0000,1.000,False
CHOL,7,3,2.95,0.81,0.0530,0.265,False
PROT,0,1,0.00,0.27,1.0000,1.000,False


### Findings: missing values by sex

Missing values are spread evenly between the sexes. 31 values are missing, 13 from the 237 women and 18 from the 371 men. No column differs after Bonferroni correction. CHOL comes closest (7 women against 3 men, adjusted p = 0.265). ALP, the only column linked to diagnosis, is missing in 2.5% of women and 3.2% of men (p = 0.81). Missing data therefore cannot explain any difference in model performance between men and women.

### Folds, imputers and models

The folds are made once and reused, so every method is tested on the same patients. Both models use balanced class weights.

In [12]:
# create the five folds once. Every method below reuses these exact
# splits, so all methods are tested on the same patients
splits = list(
    StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    .split(features, target)
)

# the three ways of filling missing values that are compared
imputers = {
    "Median": SimpleImputer(strategy="median"),
    "Median + flags": SimpleImputer(strategy="median", add_indicator=True),
    "Iterative": IterativeImputer(
        initial_strategy="median", max_iter=50, tol=0.001,
        min_value=0, random_state=SEED,
    ),
}

# the two classifiers. Class weights are balanced because the
# disease groups are small
models = {
    "Logistic Regression": LogisticRegression(
        C=1, class_weight="balanced", max_iter=5000, random_state=SEED,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, max_features="sqrt",
        class_weight="balanced", random_state=SEED, n_jobs=-1,
    ),
}

# names of the scores that are calculated for every run
METRICS = ["Macro F1", "Balanced accuracy"]
METRICS += [name + " recall" for name in CLASS_ORDER]

### Run the comparison

The imputer, scaler and encoder are inside the pipeline, so they only learn from the training folds.

In [13]:
score_rows = []
predictions_by_run = {}

for model_name, model in models.items():
    for method_name, imputer in imputers.items():
        # preprocessing is fitted on the training folds only: fill missing
        # values, scale the numbers and encode Sex, then fit the model
        numeric_steps = Pipeline([
            ("imputer", clone(imputer)),
            ("scaler", StandardScaler()),
        ])
        preprocessing = ColumnTransformer([
            ("numeric", numeric_steps, NUMERIC_COLUMNS),
            ("sex", OneHotEncoder(handle_unknown="ignore"), ["Sex"]),
        ])
        pipeline = Pipeline([
            ("preprocessing", preprocessing),
            ("model", clone(model)),
        ])

        # Out-of-fold predictions: each patient is predicted by a model
        # that never saw that patient during training
        predictions = pd.Series(index=target.index, dtype=object)

        for fold, (train, test) in enumerate(splits, start=1):
            fitted = clone(pipeline).fit(
                features.iloc[train], target.iloc[train]
            )
            predicted = fitted.predict(features.iloc[test])
            actual = target.iloc[test]
            predictions.iloc[test] = predicted

            # scores for this fold
            scores = {
                "Model": model_name,
                "Imputation": method_name,
                "Fold": fold,
                "Macro F1": f1_score(actual, predicted, average="macro"),
                "Balanced accuracy": balanced_accuracy_score(
                    actual, predicted
                ),
            }
            # recall of each diagnosis separately (share correctly found)
            recalls = recall_score(
                actual, predicted, labels=CLASS_ORDER, average=None,
                zero_division=0,
            )
            for name, value in zip(CLASS_ORDER, recalls):
                scores[name + " recall"] = value
            score_rows.append(scores)

        predictions_by_run[(model_name, method_name)] = predictions

results = pd.DataFrame(score_rows)

### Mean scores and fold variation

Mean and standard deviation over the five folds. Each fold has only four or five hepatitis and fibrosis patients, so one mistake changes their recall a lot.

In [14]:
# mean and standard deviation over the five folds for each combination
summary = results.groupby(["Model", "Imputation"])[METRICS].agg(
    ["mean", "std"]
)

# combine them into one readable "mean ± sd" text per cell
comparison = pd.DataFrame(index=summary.index)
for metric in METRICS:
    comparison[metric] = (
        summary[(metric, "mean")].map("{:.3f}".format)
        + " ± "
        + summary[(metric, "std")].map("{:.3f}".format)
    )

# overall scores first, then recall for each diagnosis
display(comparison[["Macro F1", "Balanced accuracy"]])
display(comparison[[name + " recall" for name in CLASS_ORDER]])

Macro F1 Balanced accuracy
Model               Imputation                                     
Logistic Regression Iterative       0.600 ± 0.058     0.678 ± 0.066
                    Median          0.612 ± 0.061     0.692 ± 0.057
                    Median + flags  0.647 ± 0.020     0.686 ± 0.035
Random Forest       Iterative       0.662 ± 0.128     0.611 ± 0.129
                    Median          0.647 ± 0.140     0.598 ± 0.141
                    Median + flags  0.660 ± 0.155     0.619 ± 0.161

Healthy recall Hepatitis recall  \
Model               Imputation                                       
Logistic Regression Iterative       0.919 ± 0.055    0.680 ± 0.228   
                    Median          0.931 ± 0.040    0.640 ± 0.261   
                    Median + flags  0.974 ± 0.008    0.640 ± 0.297   
Random Forest       Iterative       1.000 ± 0.000    0.370 ± 0.254   
                    Median          1.000 ± 0.000    0.320 ± 0.303   
                    Median + flags  1.000 ± 0.000    0.320 ± 0.268   

                                   Fibrosis recall Cirrhosis recall  
Model               Imputation                                       
Logistic Regression Iterative        0.480 ± 0.254    0.633 ± 0.274  
                    Median           0.530 ± 0.220    0.667 ± 0.312  
                    Median + flags   0.430 ± 0.208    0.700 ± 0.274  
Random Forest       Iterative        0.340 ± 0.290    0.733 ± 0.190  
                    Median           0.340 ± 0.290    0.733 ± 0.190  
                    Median + flags   0.390 ± 0.342    0.767 ± 0.190

### Difference from median imputation

Positive values mean the other method scored higher than the median on the same folds.

In [15]:
# mean score of every model and imputation method across the folds
means = results.groupby(["Model", "Imputation"])[METRICS].mean()

# subtract the median-imputation scores of the same model
rows = []
for model_name in models:
    median_means = means.loc[(model_name, "Median")]
    for method_name in ["Median + flags", "Iterative"]:
        change = means.loc[(model_name, method_name)] - median_means
        rows.append(change.rename((model_name, method_name)))

differences = pd.DataFrame(rows)
differences.index.names = ["Model", "Compared with median"]
differences.round(3)

Macro F1  Balanced accuracy  \
Model               Compared with median                                
Logistic Regression Median + flags           0.035             -0.006   
                    Iterative               -0.012             -0.014   
Random Forest       Median + flags           0.013              0.021   
                    Iterative                0.015              0.012   

                                          Healthy recall  Hepatitis recall  \
Model               Compared with median                                     
Logistic Regression Median + flags                 0.043              0.00   
                    Iterative                     -0.011              0.04   
Random Forest       Median + flags                 0.000              0.00   
                    Iterative                      0.000              0.05   

                                          Fibrosis recall  Cirrhosis recall  
Model               Compared with median                                     
Logistic Regression Median + flags                  -0.10             0.033  
                    Iterative                       -0.05            -0.033  
Random Forest       Median + flags                   0.05             0.033  
                    Iterative                        0.00             0.000

### Held-out mistakes

Confusion matrices over all folds. Rows are the real diagnosis and columns are the prediction.

In [16]:
# one confusion matrix per model and imputation method
for (model_name, method_name), predictions in predictions_by_run.items():
    # rows are the actual diagnosis, columns the predicted one
    confusion = pd.crosstab(target, predictions).reindex(
        index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0
    )
    print(f"{model_name} - {method_name}")
    display(confusion)

Logistic Regression - Median


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,496,20,10,7
Hepatitis,2,15,5,2
Fibrosis,1,7,11,2
Cirrhosis,1,1,8,20


Logistic Regression - Median + flags


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,519,5,6,3
Hepatitis,2,15,5,2
Fibrosis,2,7,9,3
Cirrhosis,1,1,7,21


Logistic Regression - Iterative


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,490,21,15,7
Hepatitis,2,16,3,3
Fibrosis,2,6,10,3
Cirrhosis,2,3,6,19


... (2 outputs left out)

Random Forest - Median + flags


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,533,0,0,0
Hepatitis,11,8,4,1
Fibrosis,7,4,8,2
Cirrhosis,4,1,2,23


Random Forest - Iterative


col_0,Healthy,Hepatitis,Fibrosis,Cirrhosis
Category,,,,
Healthy,533,0,0,0
Hepatitis,11,9,2,2
Fibrosis,7,5,7,2
Cirrhosis,6,0,2,22


### Limits of the comparison

This is one five-fold split with fixed model settings, so it cannot prove that one method is better. ALP is missing mostly in sick patients, so a missing-value flag may only show which tests were ordered. Shadbahr et al. (2023) found that no imputation method was best across datasets, and Sisk et al. (2023) found that missing-value flags only help under some patterns of missing data.

- [Sisk et al. (2023)](https://doi.org/10.1177/09622802231165001)
- [Shadbahr et al. (2023)](https://doi.org/10.1038/s43856-023-00356-z)

### Findings: imputation comparison

No method is best on every metric. With logistic regression, the median gives macro F1 0.612, balanced accuracy 0.692 and fibrosis recall 0.530. The median with flags gives 0.647, 0.686 and 0.430. Iterative imputation gives 0.600, 0.678 and 0.480. Random forest with iterative imputation has the highest macro F1 (0.662) but a fibrosis recall of only 0.340.

The differences are small and no method wins on every metric, so the median is used because it is the simplest.

**Decision:** fill the missing values with the median.

## Apply median imputation

The missing values of the 608 patients are filled with the column median. The median is taken from all patients, so models that are cross-validated later should start from the unimputed data and impute inside each training fold.

In [17]:
# fill the gaps in the numeric columns with each column's median
# Category and Sex are not changed
df_imputed = comparison_df.copy()
df_imputed[NUMERIC_COLUMNS] = SimpleImputer(strategy="median").fit_transform(
    comparison_df[NUMERIC_COLUMNS]
)

# every count should now be zero
df_imputed.isna().sum().rename("Missing after imputation")

Category    0
Age         0
Sex         0
ALB         0
ALP         0
ALT         0
AST         0
BIL         0
CHE         0
CHOL        0
CREA        0
GGT         0
PROT        0
cat_num     0
Name: Missing after imputation, dtype: int64

## Outlier detection (report only)

Outliers are found in two ways. The IQR rule flags a value that is more than 1.5 times the IQR below the first quartile or above the third quartile. The Isolation Forest (100 trees, 256 samples per tree, contamination 0.05) flags rows that are unusual across all eleven variables. Nothing is removed.

In [18]:
# outliers are judged on the imputed measurements only (no Sex)
numeric_data = df_imputed[NUMERIC_COLUMNS]

# the IQR is the spread of the middle half of the values
q1 = numeric_data.quantile(0.25)
q3 = numeric_data.quantile(0.75)
iqr = q3 - q1

# A value is an outlier if it is more than 1.5 IQR below Q1 or above Q3
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
iqr_outliers = (numeric_data < lower) | (numeric_data > upper)

# number of outlier values in each column
iqr_outliers.sum().rename("IQR outlier values")

Age      1
ALB     22
ALP     12
ALT     34
AST     61
BIL     46
CHE     23
CHOL    12
CREA    10
GGT     68
PROT    15
Name: IQR outlier values, dtype: int64

In [19]:
# Isolation Forest looks at all 11 variables together and flags the
# 5% of rows that are easiest to separate from the rest
forest = IsolationForest(
    n_estimators=100, max_samples=256, contamination=0.05,
    random_state=SEED,
)

# fit_predict returns -1 for an outlier and 1 for a normal row
forest_outliers = pd.Series(
    forest.fit_predict(numeric_data) == -1, index=df_imputed.index
)

print("Rows flagged by Isolation Forest:", forest_outliers.sum())

Rows flagged by Isolation Forest: 31


### Flagged rows by category

Number and percentage of rows flagged in each category.

In [20]:
# count flagged rows in each category: rows with at least one IQR
# outlier, and rows flagged by the Isolation Forest
flagged = pd.DataFrame({
    "Patients": df_imputed["Category"].value_counts(),
    "IQR rows": iqr_outliers.any(axis=1).groupby(df_imputed["Category"]).sum(),
    "Isolation Forest rows": forest_outliers.groupby(
        df_imputed["Category"]
    ).sum(),
}).reindex(CLASS_ORDER)

# express each count as a percentage of the category size
flagged["IQR (%)"] = (100 * flagged["IQR rows"] / flagged["Patients"]).round(1)
flagged["Isolation Forest (%)"] = (
    100 * flagged["Isolation Forest rows"] / flagged["Patients"]
).round(1)

flagged

,Patients,IQR rows,Isolation Forest rows,IQR (%),Isolation Forest (%)
Category,,,,,
Healthy,533,104,3,19.5,0.6
Hepatitis,24,15,4,62.5,16.7
Fibrosis,21,17,2,81.0,9.5
Cirrhosis,30,30,22,100.0,73.3


### Findings: outliers

The IQR rule flags 166 of the 608 patients (27%). Most outliers are in GGT (68), AST (61), BIL (46) and ALT (34). All 30 cirrhosis patients, 17 of 21 fibrosis patients and 15 of 24 hepatitis patients have at least one outlier, against 104 of 533 healthy patients. The Isolation Forest flags 31 rows and 28 of them are sick patients.

**No rows are removed.** High liver enzymes and bilirubin are signs of liver disease, so these values are real. Removing them would delete most of the sick patients.

## Save the cleaned data

The imputed dataset is saved without the index.

In [21]:
# write the cleaned data. index=False avoids an extra index column
# no outlier flags are added
df_imputed.to_csv(OUTPUT_PATH, index=False)